# 🎬 TeliCinema — Telegram Hardcoded Subtitle Bot (Google Colab Edition)

بوت احترافي لدمج الترجمات العربية بشكل دائم ومباشر (Hardsub) داخل الفيديوهات عبر **FFmpeg** مع دعم محرك التشكيل **HarfBuzz** وخط **Amiri** المعتمد.

---

### الخلية 1 — تثبيت الاعتماديات والنظام والخطوط العربية
تثبيت أحدث حزم FFmpeg وخطوط Naskh العربية (Amiri وNoto Sans Arabic) والمكتبات البرمجية.

In [ ]:
# Install System Dependencies, FFmpeg and Arabic Typefaces
!apt-get update -qq
!apt-get install -y -qq ffmpeg fonts-amiri fonts-noto-core fonts-dejavu-core

# Install Python MTProto Client & Utilities
!pip install --upgrade -q hydrogram tgcrypto pysubs2 nest-asyncio python-dotenv psutil

### الخلية 2 — الإعداد والتكوين الآمن
إدخال مفاتيح الاعتماد الخاصة بتيليجرام بشكل آمن دون كتابتها صراحة في الكود.

In [ ]:
import os
import getpass

print("🔐 أدخل بيانات الاعتماد الخاصة بحسابك وبوت تيليجرام:")

api_id = input("1. أدخل Telegram API ID: ").strip()
api_hash = getpass.getpass("2. أدخل Telegram API Hash: ").strip()
bot_token = getpass.getpass("3. أدخل Telegram Bot Token: ").strip()
owner_id = input("4. أدخل معرف المالك (Telegram Owner User ID): ").strip()
group_id = input("5. أدخل معرف المجموعة الخاصة (Group ID e.g. -100xxxx): ").strip()

# Verification
assert api_id.isdigit(), "❌ يجب أن يكون API ID رقماً صحيحاً!"
assert len(api_hash) >= 16, "❌ تأكد من صحة API Hash!"
assert ":" in bot_token, "❌ تأكد من صحة رمز البوت (Bot Token)!"
assert owner_id.lstrip('-').isdigit(), "❌ يجب أن يكون Owner ID رقماً صحيحاً!"
assert group_id.lstrip('-').isdigit(), "❌ يجب أن يكون Group ID رقماً صحيحاً!"

os.environ["TELEGRAM_API_ID"] = api_id
os.environ["TELEGRAM_API_HASH"] = api_hash
os.environ["TELEGRAM_BOT_TOKEN"] = bot_token
os.environ["TELEGRAM_OWNER_ID"] = owner_id
os.environ["TELEGRAM_GROUP_ID"] = group_id

print("\n✅ تم التحقق من صحة المدخلات وتخزينها في متغيرات البيئة بأمان.")

### الخلية 3 — إنشاء وحدة الإعدادات (`config.py`)

In [ ]:
%%writefile config.py
from __future__ import annotations
import os
import shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import Set

@dataclass(frozen=True)
class Config:
    API_ID: int
    API_HASH: str
    BOT_TOKEN: str
    OWNER_ID: int
    GROUP_ID: int

    BASE_DIR: Path = Path("/content/telicinema_workspace")
    JOBS_DIR: Path = Path("/content/telicinema_workspace/jobs")
    SESSIONS_DIR: Path = Path("/content/telicinema_workspace/sessions")

    USER_SESSION_NAME: str = "telicinema_user"
    BOT_SESSION_NAME: str = "telicinema_bot"

    SUPPORTED_VIDEO_EXTS: Set[str] = field(
        default_factory=lambda: {".mp4", ".mkv", ".mov", ".avi", ".webm", ".m4v", ".flv", ".ts"}
    )
    SUPPORTED_SUB_EXTS: Set[str] = field(
        default_factory=lambda: {".srt", ".vtt", ".ass"}
    )

    FONT_NAME: str = "Amiri"
    FONT_SIZE: int = 50
    ALIGNMENT: int = 2
    FFMPEG_PRESET: str = "veryfast"
    FFMPEG_CRF: int = 22
    AUDIO_CODEC: str = "aac"
    AUDIO_BITRATE: str = "192k"

    @classmethod
    def from_env(cls) -> Config:
        api_id_raw = os.getenv("TELEGRAM_API_ID", "").strip()
        api_hash = os.getenv("TELEGRAM_API_HASH", "").strip()
        bot_token = os.getenv("TELEGRAM_BOT_TOKEN", "").strip()
        owner_id_raw = os.getenv("TELEGRAM_OWNER_ID", "").strip()
        group_id_raw = os.getenv("TELEGRAM_GROUP_ID", "").strip()

        config = cls(
            API_ID=int(api_id_raw),
            API_HASH=api_hash,
            BOT_TOKEN=bot_token,
            OWNER_ID=int(owner_id_raw),
            GROUP_ID=int(group_id_raw),
        )
        config.setup_directories()
        return config

    def setup_directories(self) -> None:
        self.BASE_DIR.mkdir(parents=True, exist_ok=True)
        self.JOBS_DIR.mkdir(parents=True, exist_ok=True)
        self.SESSIONS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG: Config | None = None

def get_config() -> Config:
    global CONFIG
    if CONFIG is None:
        CONFIG = Config.from_env()
    return CONFIG

### الخلية 4 — إنشاء معالج الترجمة والفيديو (`subtitle_processor.py`)

In [ ]:
%%writefile subtitle_processor.py
from __future__ import annotations
import asyncio
import os
import re
import subprocess
from pathlib import Path
from typing import Callable, Optional, Dict, Any
import pysubs2
from config import get_config

class SubtitleProcessorError(Exception):
    pass

class SubtitleProcessor:
    def __init__(self, job_dir: Path):
        self.job_dir = job_dir
        self.cfg = get_config()
        self._current_process: Optional[asyncio.subprocess.Process] = None
        self._is_cancelled: bool = False

    @staticmethod
    def detect_and_read_subtitle(file_path: Path) -> str:
        encodings = ["utf-8-sig", "utf-8", "cp1256", "iso-8859-6", "latin1"]
        raw_bytes = file_path.read_bytes()
        for enc in encodings:
            try:
                content = raw_bytes.decode(enc)
                if "-->" in content:
                    return content
            except UnicodeDecodeError:
                continue
        raise SubtitleProcessorError(f"Unable to decode subtitle file '{file_path.name}'.")

    def convert_srt_to_ass(self, srt_path: Path, video_width: int = 1920, video_height: int = 1080) -> Path:
        ass_path = self.job_dir / f"{srt_path.stem}.ass"
        text = self.detect_and_read_subtitle(srt_path)
        subs = pysubs2.SSAFile.from_string(text)

        subs.info["PlayResX"] = str(video_width)
        subs.info["PlayResY"] = str(video_height)
        subs.info["WrapStyle"] = "0"
        subs.info["ScaledBorderAndShadow"] = "yes"

        calc_fontsize = max(24, int(video_height * 0.046))
        calc_margin_v = max(20, int(video_height * 0.040))

        arabic_style = pysubs2.SSAStyle(
            fontname=self.cfg.FONT_NAME,
            fontsize=calc_fontsize,
            primarycolor=pysubs2.Color(255, 255, 255, 0),
            outlinecolor=pysubs2.Color(0, 0, 0, 0),
            backcolor=pysubs2.Color(0, 0, 0, 128),
            bold=True,
            borderstyle=1,
            outline=round(calc_fontsize * 0.055, 1),
            shadow=round(calc_fontsize * 0.025, 1),
            alignment=self.cfg.ALIGNMENT,
            marginv=calc_margin_v,
            encoding=-1
        )
        subs.styles["ArabicDefault"] = arabic_style
        for event in subs.events:
            if not event.is_comment:
                event.style = "ArabicDefault"
                event.text = event.text.replace("\r\n", r"\N").replace("\n", r"\N")

        subs.save(str(ass_path), encoding="utf-8")
        return ass_path

    async def probe_video_stream(self, video_path: Path) -> Dict[str, Any]:
        cmd = [
            "ffprobe", "-v", "error", "-select_streams", "v:0",
            "-show_entries", "stream=width,height,duration:format=duration",
            "-of", "default=noprint_wrappers=1", str(video_path)
        ]
        proc = await asyncio.create_subprocess_exec(*cmd, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
        stdout, _ = await proc.communicate()
        data = {}
        for line in stdout.decode().splitlines():
            if "=" in line:
                k, v = line.split("=", 1)
                data[k.strip()] = v.strip()
        return {
            "width": int(data.get("width", 1920)),
            "height": int(data.get("height", 1080)),
            "duration": float(data.get("duration", 0.0) or 0.0)
        }

    async def burn_subtitles(self, video_path: Path, srt_path: Path, output_path: Path, progress_callback: Optional[Callable] = None) -> Path:
        meta = await self.probe_video_stream(video_path)
        ass_path = self.convert_srt_to_ass(srt_path, meta["width"], meta["height"])
        escaped_ass = str(ass_path.resolve()).replace("\\", "/").replace(":", r"\:").replace("'", r"\'")
        
        ass_filter = f"ass='{escaped_ass}':shaping=complex"

        cmd = [
            "ffmpeg", "-y", "-i", str(video_path),
            "-vf", ass_filter,
            "-c:v", "libx264", "-preset", self.cfg.FFMPEG_PRESET, "-crf", str(self.cfg.FFMPEG_CRF),
            "-pix_fmt", "yuv420p", "-c:a", self.cfg.AUDIO_CODEC, "-b:a", self.cfg.AUDIO_BITRATE,
            "-movflags", "+faststart", "-progress", "pipe:1", "-nostats", str(output_path)
        ]

        self._current_process = await asyncio.create_subprocess_exec(*cmd, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
        time_pattern = re.compile(r"out_time_ms=(\d+)")

        while True:
            line = await self._current_process.stdout.readline()
            if not line:
                break
            match = time_pattern.search(line.decode(errors="replace"))
            if match and meta["duration"] > 0 and progress_callback:
                cur_s = int(match.group(1)) / 1_000_000.0
                pct = min(99.0, (cur_s / meta["duration"]) * 100.0)
                res = progress_callback(pct, f"{pct:.1f}% ({int(cur_s)}s / {int(meta['duration'])}s)")
                if asyncio.iscoroutine(res):
                    await res

        stderr_out = await self._current_process.stderr.read()
        await self._current_process.wait()

        if self._current_process.returncode != 0:
            if output_path.exists(): output_path.unlink()
            raise SubtitleProcessorError(f"FFmpeg Error: {stderr_out.decode(errors='replace')[-400:]}")
        return output_path

    def cancel(self):
        if self._current_process:
            try: self._current_process.terminate()
            except ProcessLookupError: pass

### الخلية 5 — إنشاء وحدة إدارة البوت وقائمة المهام (`bot.py`)

In [ ]:
%%writefile bot.py
from __future__ import annotations
import asyncio
import shutil
import time
import uuid
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional
from hydrogram import Client, filters
from hydrogram.types import Message
from config import get_config
from subtitle_processor import SubtitleProcessor

@dataclass
class SubtitleJob:
    job_id: str
    group_id: int
    video_message_id: int
    subtitle_message_id: int
    video_file_name: str
    subtitle_file_name: str
    status_message_id: Optional[int] = None
    processor: Optional[SubtitleProcessor] = None
    job_dir: Optional[Path] = None

class TeliCinemaBot:
    def __init__(self):
        self.cfg = get_config()
        self.queue: asyncio.Queue[SubtitleJob] = asyncio.Queue()
        self.active_job: Optional[SubtitleJob] = None
        self.pending_videos: Dict[int, Message] = {}
        self.is_running: bool = False

        self.bot_client = Client(
            name=str(self.cfg.SESSIONS_DIR / self.cfg.BOT_SESSION_NAME),
            api_id=self.cfg.API_ID,
            api_hash=self.cfg.API_HASH,
            bot_token=self.cfg.BOT_TOKEN,
        )
        self.user_client = Client(
            name=str(self.cfg.SESSIONS_DIR / self.cfg.USER_SESSION_NAME),
            api_id=self.cfg.API_ID,
            api_hash=self.cfg.API_HASH,
        )
        self._setup_handlers()

    def _setup_handlers(self):
        @self.bot_client.on_message(filters.command("start") & filters.private)
        async def on_start(_, msg: Message):
            if msg.from_user.id == self.cfg.OWNER_ID:
                await msg.reply_text("🎬 **مرحباً بك في TeliCinema!**\nأرسل الفيديو في المجموعة الخاصة، ثم رد عليه بملف الترجمة .srt.")

        @self.bot_client.on_message(filters.command("status") & filters.private)
        async def on_status(_, msg: Message):
            if msg.from_user.id == self.cfg.OWNER_ID:
                await msg.reply_text(f"📊 المهام في الانتظار: {self.queue.qsize()}\nالمهمة الحالية: {self.active_job.job_id if self.active_job else 'لا توجد'}")

        @self.bot_client.on_message(filters.chat(self.cfg.GROUP_ID) & (filters.video | filters.document))
        async def on_group_media(_, msg: Message):
            if msg.document and msg.reply_to_message_id:
                fname = msg.document.file_name or ""
                if fname.lower().endswith(tuple(self.cfg.SUPPORTED_SUB_EXTS)):
                    parent = self.pending_videos.get(msg.reply_to_message_id)
                    if not parent:
                        parent = await self.bot_client.get_messages(self.cfg.GROUP_ID, msg.reply_to_message_id)
                    if parent and (parent.video or parent.document):
                        jid = str(uuid.uuid4())[:8]
                        job = SubtitleJob(
                            job_id=jid,
                            group_id=self.cfg.GROUP_ID,
                            video_message_id=parent.id,
                            subtitle_message_id=msg.id,
                            video_file_name=parent.video.file_name if parent.video else parent.document.file_name or "video.mp4",
                            subtitle_file_name=fname,
                        )
                        ack = await msg.reply_text(f"⏳ تم ربط الترجمة بالفيديو. المعرف: `{jid}`. تمت الإضافة لقائمة الانتظار...")
                        job.status_message_id = ack.id
                        await self.queue.put(job)
                        return
            if msg.video or (msg.document and any((msg.document.file_name or "").lower().endswith(x) for x in self.cfg.SUPPORTED_VIDEO_EXTS)):
                self.pending_videos[msg.id] = msg
                await msg.reply_text("✅ تم استلام الفيديو. أرسل ملف الترجمة (.srt) كردّ مباشر (Reply) على هذه الرسالة.")

    async def _update_status(self, job: SubtitleJob, text: str):
        if job.status_message_id:
            try: await self.bot_client.edit_message_text(job.group_id, job.status_message_id, text)
            except Exception: pass

    async def _worker(self):
        while self.is_running:
            job = await self.queue.get()
            self.active_job = job
            job_dir = self.cfg.JOBS_DIR / job.job_id
            job.job_dir = job_dir
            job_dir.mkdir(parents=True, exist_ok=True)

            try:
                await self._update_status(job, "⏳ جارٍ تنزيل الفيديو والترجمة...")
                v_msg = await self.bot_client.get_messages(job.group_id, job.video_message_id)
                s_msg = await self.bot_client.get_messages(job.group_id, job.subtitle_message_id)

                v_path = job_dir / f"v{Path(job.video_file_name).suffix or '.mp4'}"
                s_path = job_dir / f"s{Path(job.subtitle_file_name).suffix or '.srt'}"
                out_path = job_dir / f"HardSub_{Path(job.video_file_name).stem}.mp4"

                await self.bot_client.download_media(v_msg, file_name=str(v_path))
                await self.bot_client.download_media(s_msg, file_name=str(s_path))

                await self._update_status(job, "🎬 جارٍ دمج الترجمة العربية داخل الفيديو...")
                proc = SubtitleProcessor(job_dir)
                job.processor = proc
                await proc.burn_subtitles(v_path, s_path, out_path)

                await self._update_status(job, "📤 اكتملت المعالجة، جارٍ إرسال الفيديو الناتج...")
                caption = f"✅ تم دمج الترجمة بنجاح.\nالملف: `{out_path.name}`"
                try:
                    await self.bot_client.send_video(self.cfg.OWNER_ID, str(out_path), caption=caption, supports_streaming=True)
                except Exception:
                    await self.user_client.send_video(self.cfg.OWNER_ID, str(out_path), caption=caption, supports_streaming=True)

                await self._update_status(job, "✅ تم دمج الترجمة وإرسال الفيديو للخاص بنجاح.")
            except Exception as e:
                await self._update_status(job, f"❌ حدث خطأ أثناء المعالجة: {str(e)}")
            finally:
                shutil.rmtree(job_dir, ignore_errors=True)
                self.pending_videos.pop(job.video_message_id, None)
                self.active_job = None
                self.queue.task_done()

    async def start(self):
        self.is_running = True
        await self.bot_client.start()
        await self.user_client.start()
        print("🚀 خادم البوت وعميل المستخدم متصلان بنجاح!")
        await self._worker()

### الخلية 6 — توثيق وتسجيل دخول حساب تيليجرام الشخصي (User Client Authentication)
تسجيل الدخول التفاعلي لإنشاء جلسة MTProto للحساب الشخصي مع دعم كود التحقق وكلمة سر التحقق بخطوتين (2FA).

In [ ]:
import asyncio
import getpass
from hydrogram import Client
from hydrogram.errors import SessionPasswordNeeded
from config import get_config

cfg = get_config()
session_path = cfg.SESSIONS_DIR / cfg.USER_SESSION_NAME

async def login_user():
    app = Client(name=str(session_path), api_id=cfg.API_ID, api_hash=cfg.API_HASH)
    await app.connect()
    
    if not await app.get_me():
        print("📲 بدء جلسة تسجيل دخول جديدة لحسابك الشخصي:")
        phone = input("أدخل رقم الهاتف مع الرمز الدولي (e.g. +966xxxxxxxxx): ").strip()
        sent_code = await app.send_code(phone)
        phone_code = getpass.getpass("أدخل كود التحقق المرسل إلى تطبيق تيليجرام: ").strip()
        
        try:
            await app.sign_in(phone, sent_code.phone_code_hash, phone_code)
        except SessionPasswordNeeded:
            pwd = getpass.getpass("الحساب محمي بالتحقق بخطوتين؛ أدخل كلمة السر (2FA): ").strip()
            await app.check_password(pwd)
            
    user = await app.get_me()
    print(f"\n✅ تم تسجيل الدخول بنجاح للحساب: {user.first_name} (ID: {user.id})")
    await app.disconnect()

await login_user()

### الخلية 7 — تشغيل البوت والاستماع للمهام
تشغيل حلقة الأحداث (Event Loop) المتوافقة مع Colab لبدء استقبال الفيديو والترجمة.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from bot import TeliCinemaBot

bot_app = TeliCinemaBot()
print("🟢 البوت قيد التشغيل الآن ويستمع للرسائل داخل المجموعة المحددة...")
await bot_app.start()